In [1]:
import pandas as pd

pd.__version__

'1.4.2'

In [2]:
from pathlib import Path

DATA_DIR = Path("../data")

df_01 = pd.read_parquet(DATA_DIR / "yellow_tripdata_2023-01.parquet")
df_02 = pd.read_parquet(DATA_DIR / "yellow_tripdata_2023-02.parquet")


In [3]:
import sklearn
sklearn.__version__

'1.0.2'

Q1. Downloading the data

We'll use the same NYC taxi dataset, but instead of "Green Taxi Trip Records", we'll use "Yellow Taxi Trip Records".

Download the data for January and February 2023.

Read the data for January. How many columns are there?

16
17
18
19

In [4]:
len(df_01.columns)

19

Q2. Computing duration

Now let's compute the duration variable. It should contain the duration of a ride in minutes.

What's the standard deviation of the trips duration in January?

32.59
42.59
52.59
62.59

In [5]:
df_01["tpep_pickup_datetime"] = pd.to_datetime(df_01["tpep_pickup_datetime"])
df_01["tpep_dropoff_datetime"] = pd.to_datetime(df_01["tpep_dropoff_datetime"])

df_01["duration"] = (df_01["tpep_dropoff_datetime"] - df_01["tpep_pickup_datetime"]).dt.total_seconds() / 60

In [6]:
import numpy as np

np.std(df_01["duration"].values)

42.59434429744777

Q3. Dropping outliers

Next, we need to check the distribution of the duration variable. There are some outliers. Let's remove them and keep only the records where the duration was between 1 and 60 minutes (inclusive).

What fraction of the records left after you dropped the outliers?

90%
92%
95%
98%

In [7]:
df_01[(df_01["duration"] >= 1) & (df_01["duration"] <= 60)].shape[0] / df_01.shape[0] * 100
df_01 = df_01[(df_01["duration"] >= 1) & (df_01["duration"] <= 60)]

Q4. One-hot encoding

Let's apply one-hot encoding to the pickup and dropoff location IDs. We'll use only these two features for our model.

Turn the dataframe into a list of dictionaries (remember to re-cast the ids to strings - otherwise it will label encode them)
Fit a dictionary vectorizer
Get a feature matrix from it
What's the dimensionality of this matrix (number of columns)?

2
155
345
515
715

In [8]:
from sklearn.feature_extraction import DictVectorizer

categorical = ["PULocationID", "DOLocationID"]

df_cat = df_01[categorical].astype(str)

train_dicts = (
    {
        "PULocationID": pu,
        "DOLocationID": do
    }
    for pu, do in zip(
        df_cat["PULocationID"],
        df_cat["DOLocationID"]
    )
)

dv = DictVectorizer()

X_train = dv.fit_transform(train_dicts)

X_train.shape

(3009173, 515)

Q5. Training a model

Now let's use the feature matrix from the previous step to train a model.

Train a plain linear regression model with default parameters, where duration is the response variable
Calculate the RMSE of the model on the training data
What's the RMSE on train?

3.64
7.64
11.64
16.64

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

y_train = df_01["duration"].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_train)

rmse = mean_squared_error(y_train, y_pred, squared=False)

rmse

7.649261027826866

Q6. Evaluating the model

Now let's apply this model to the validation dataset (February 2023).

What's the RMSE on validation?

3.81
7.81
11.81
16.81

In [10]:
df_02["tpep_pickup_datetime"] = pd.to_datetime(df_02["tpep_pickup_datetime"])
df_02["tpep_dropoff_datetime"] = pd.to_datetime(df_02["tpep_dropoff_datetime"])

df_02["duration"] = (df_02["tpep_dropoff_datetime"] - df_02["tpep_pickup_datetime"]).dt.total_seconds() / 60

In [11]:
df_02 = df_02[(df_02["duration"] >= 1) & (df_02["duration"] <= 60)]

In [12]:
df02_cat = df_02[categorical].astype(str)

test_dicts = (
    {"PULocationID": pu, "DOLocationID": do}
    for pu, do in zip(df02_cat["PULocationID"], df02_cat["DOLocationID"])
)
X_val = dv.transform(test_dicts)

In [14]:
y_val = df_02["duration"].values

y_pred = lr.predict(X_val)

rmse = mean_squared_error(y_val, y_pred, squared=False)

rmse

7.811832641626525